# H1 EDA: 성장 정체와 접속 행동

현재 H1 입력 스키마, 결측, 성장 정체 비율, 주간 접속 관측 품질을 점검한다.
성장 정체 군집은 후보 확정 라벨이 아니며, 접속 행동은 외부 검증 단계에서 결합한다.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA = Path('../data')
features = pd.read_csv(DATA / 'features_monthly.csv', encoding='utf-8-sig')
raw = pd.read_csv(DATA / 'monthly_snapshots_raw.csv', encoding='utf-8-sig')
hexa = pd.read_csv(DATA / 'hexa_fragments.csv', encoding='utf-8-sig')

print('features:', features.shape, '| raw:', raw.shape, '| hexa:', hexa.shape)
print('raw 기간:', raw['year_month'].min(), '~', raw['year_month'].max())
assert {'access_active_weeks', 'access_observed_weeks'}.issubset(raw.columns)

features: (2000, 44) | raw: (24000, 12) | hexa: (2000, 23)
raw 기간: 2025-06 ~ 2026-05


## 1. H1 피처 품질

전체 기간 성장 정체 군집 탐색에 사용하는 세 피처의 결측과 분포를 확인한다.

In [2]:
df = features.merge(hexa[['ocid', 'avg_monthly_delta_hexa_frag']], on='ocid', how='left')
df = df[df['level'].between(270, 290)].copy()
df['cumexp'] = df['log1p_avg_monthly_delta_cumexp'].clip(lower=0)
df['union'] = df['avg_monthly_delta_union_level'].clip(lower=0)
df['hfrag'] = df['avg_monthly_delta_hexa_frag'].clip(lower=0)
cols = ['cumexp', 'union', 'hfrag']
print('분석 표본:', len(df))
print('결측:')
print(df[cols].isna().sum().to_string())
print('\n분포:')
display(df[cols].describe().round(3))
print('\n0 비율:')
print((df[cols].eq(0).mean() * 100).round(1).astype(str).add('%').to_string())

분석 표본: 1988
결측:
cumexp     4
union     21
hfrag      0

분포:


,cumexp,union,hfrag
count,1984.000,1967.000,1988.000
mean,23.017,44.700,145.989
std,11.927,94.468,227.599
min,0.000,0.000,0.000
25%,22.092,0.182,5.091
50%,28.987,18.636,60.273
75%,30.900,41.818,191.341
max,33.133,814.400,3428.000



0 비율:
cumexp    19.8%
union     22.4%
hfrag     23.4%


## 2. 접속 관측 품질

월 1회 관측 대신 월내 4개 기준일을 조회한다. 원시 이력의 관측 주 수와 활동 주 수를 확인한다.

In [3]:
print('월별 access_flag:')
print(raw['access_flag'].value_counts(dropna=False).rename_axis('access_flag').to_string())
print('\n주간 관측 요약:')
display(raw[['access_active_weeks', 'access_observed_weeks']].describe().round(3))
print('\n캐릭터별 활동 요약:')
display(features[['access_active_months', 'access_active_weeks', 'access_observed_weeks', 'access_ratio']].describe().round(3))

월별 access_flag:
access_flag
1.0    13477
0.0     9054
NaN     1469

주간 관측 요약:


,access_active_weeks,access_observed_weeks
count,22531.000,24000.000
mean,2.005,3.727
std,1.852,0.988
min,0.000,0.000
25%,0.000,4.000
50%,2.000,4.000
75%,4.000,4.000
max,4.000,4.000



캐릭터별 활동 요약:


,access_active_months,access_active_weeks,access_observed_weeks,access_ratio
count,2000.000,2000.000,2000.000,2000.000
mean,6.738,22.584,44.725,0.500
std,4.359,18.050,7.566,0.386
min,0.000,0.000,4.000,0.000
25%,2.000,4.750,48.000,0.104
50%,7.000,20.000,48.000,0.438
75%,12.000,43.000,48.000,0.958
max,12.000,48.000,48.000,1.000


## 3. 현재 해석

- 성장 정체 군집과 주차 후보는 구분한다.
- 현재 후보는 시간 분할 노트북에서 `성장축 정체 + 반복 접속`으로 생성한다.
- H2/H3는 `data/h1_current_candidates.csv`의 후보 라벨을 명시적으로 선택한다.